# Multimodal SFT: Video Action Recognition

Fine-tune GPT-4.1 for UCF101 actions represented as ordered privacy-preserving low-frequency color frames. This notebook covers only UCF101 retrieval, clip-safe splits, preprocessing, guarded training/reuse, deployment inference, and per-class evaluation. `FOUNDRY_RUN_PAID_JOBS=false` and `FOUNDRY_RUN_LIVE_EVALUATION=false` by default.

## Setup

Copy `../.env.template` to `.env`, run `az login`, and install `../../../requirements.lock`. Kaggle downloads and generated frames/JSONL remain ignored.

In [ ]:
from __future__ import annotations
import hashlib, io, json, os, random, sys, time, urllib.request, zipfile
from collections import Counter
from pathlib import Path
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
DEMO_ROOT=Path.cwd().resolve()
if DEMO_ROOT.name=="notebooks": DEMO_ROOT=DEMO_ROOT.parent
TRACK_ROOT=DEMO_ROOT.parent
sys.path.insert(0,str(TRACK_ROOT/"src"))
from multimodal_common import assert_disjoint, image_uri, live_evaluation_enabled, paid_jobs_enabled, project_clients, resolve_or_create_job, sha256, vision_chat, write_jsonl
load_dotenv(DEMO_ROOT/".env")
ROOT=DEMO_ROOT; OUTPUTS=ROOT/"outputs"; OUTPUTS.mkdir(exist_ok=True)
FOUNDRY_RUN_PAID_JOBS=paid_jobs_enabled(); FOUNDRY_RUN_LIVE_EVALUATION=live_evaluation_enabled(); PROJECT_ENDPOINT=os.getenv("FOUNDRY_PROJECT_ENDPOINT","").strip()
print({"demo_root":str(ROOT),"run_paid_jobs":FOUNDRY_RUN_PAID_JOBS,"run_live_evaluation":FOUNDRY_RUN_LIVE_EVALUATION,"project_configured":bool(PROJECT_ENDPOINT)})

## Data contract

The source demo committed no generated UCF101 split JSONL. Reproducibility comes from the official CSV partitions, fixed top-three-class selection, seed 0, and 12/4/5 clips per class.

In [ ]:
manifest_path=ROOT/"data"/"hash-manifest.csv"
if manifest_path.exists():
    hash_manifest=pd.read_csv(manifest_path)
    for row in hash_manifest.itertuples(index=False):
        if sha256(ROOT/row.preserved_path)!=row.preserved_sha256: raise ValueError(row.preserved_path)
    display(hash_manifest[["dataset","preserved_path","split","rows","byte_match"]])
else:
    print("No committed generated JSONL existed for this source demo; live preparation is authoritative.")

## 4. Video action recognition

Download UCF101 live, retain the official train/validation/test CSVs, choose the three most common training classes, and sample 12/4/5 clips per class with seed 0. Extract three evenly spaced privacy-preserving low-frequency color frames per clip. Each frame is reduced to 8 by 8 pixels, smoothly reconstructed, and strongly blurred to remove identity detail and the text-like or block-edge patterns rejected by the live fine-tuning preprocessor. Every frame from a clip remains in one split.

In [ ]:
def privacy_frame(frame):
    import cv2
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); small=cv2.resize(rgb,(8,8),interpolation=cv2.INTER_AREA); smooth=cv2.resize(small,(256,256),interpolation=cv2.INTER_CUBIC); smooth=cv2.GaussianBlur(smooth,(31,31),0); return Image.fromarray(smooth).convert("RGB")
def extract_frames(path,count=3):
    import cv2
    cap=cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise ValueError(f"Cannot open {path}")
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total<=0: cap.release(); raise ValueError(f"No frames in {path}")
    out=[]
    for index in np.linspace(0,total-1,count,dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(index)); ok,frame=cap.read()
        if not ok: cap.release(); raise ValueError(f"Cannot read frame {index}")
        out.append(privacy_frame(frame))
    cap.release(); return out
def prepare_ucf101():
    import kagglehub
    os.environ["KAGGLEHUB_CACHE"]=str(ROOT/"data"/"downloads"/"kagglehub"); data=Path(kagglehub.dataset_download("matthewjansen/ucf101-action-recognition"))
    originals={}
    for split,name in (("train","train.csv"),("validation","val.csv"),("holdout","test.csv")):
        frame=pd.read_csv(data/name); frame["clip_path"]=frame.clip_path.str.lstrip("/"); originals[split]=frame
    classes=list(originals["train"].label.value_counts().index[:3]); sizes={"train":12,"validation":4,"holdout":5}
    frames={split:frame[frame.label.isin(classes)].groupby("label",group_keys=False).sample(n=sizes[split],random_state=0).reset_index(drop=True) for split,frame in originals.items()}; assert_disjoint({k:set(v.clip_path) for k,v in frames.items()})
    system="Identify the activity from time-ordered privacy-preserving low-frequency color frames. Choose exactly one of: "+", ".join(classes)+'. Return JSON as {"activity": "label"}.'; files={}
    for split in ("train","validation"):
        rows=[]
        for sample in frames[split].itertuples(index=False):
            images=[{"type":"image_url","image_url":{"url":image_uri(im,None,90)}} for im in extract_frames(data/sample.clip_path,3)]
            rows.append({"messages":[{"role":"system","content":system},{"role":"user","content":"These are ordered frames from one clip."},{"role":"user","content":images},{"role":"assistant","content":json.dumps({"activity":sample.label})}]})
        files[split]=write_jsonl(rows,OUTPUTS/"video-action-recognition"/f"ucf101-{split}.jsonl")
    frames["holdout"].to_csv(OUTPUTS/"video-action-recognition"/"holdout-manifest.csv",index=False); print({k:{"clips":len(v),"classes":v.label.value_counts().to_dict()} for k,v in frames.items()}); return files,frames,system,data
video_files,video_data,video_system,ucf_root=prepare_ucf101()

### Train or reuse

The exact returned train and validation files are uploaded only when the paid-job gate is deliberately enabled. Reused jobs are still monitored and must succeed. Otherwise this remains a preparation and evaluation lesson.

In [ ]:
video_job=None
video_reuse_id=os.getenv("FOUNDRY_VIDEO_FINE_TUNING_JOB_ID","").strip()
if FOUNDRY_RUN_PAID_JOBS or video_reuse_id:
    if not PROJECT_ENDPOINT: raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT to submit or monitor a fine-tuning job.")
    with project_clients(PROJECT_ENDPOINT) as (_,video_client):
        video_job=resolve_or_create_job(client=video_client,reuse_id=video_reuse_id,model="gpt-4.1-2025-04-14",train=video_files["train"],validation=video_files["validation"],suffix="multimodal-ucf101-actions",seed=0,training_type="globalStandard",run_paid_jobs=FOUNDRY_RUN_PAID_JOBS)
        if video_job:
            events=list(video_client.fine_tuning.jobs.list_events(fine_tuning_job_id=video_job.id,limit=100).data)
            preprocessing_events=[{"level":event.level,"message":event.message} for event in events if "Preprocessing Summary" in event.message]
            if preprocessing_events:
                print("Service-side preprocessing changed the usable row count; review before interpreting the model.")
                display(pd.DataFrame(preprocessing_events))
else: print("Paid training and existing-job monitoring are disabled.")

### Video inference, per-class quality, and robustness

When `FOUNDRY_RUN_LIVE_EVALUATION=true`, send ordered frames together and report accuracy, per-class precision/recall/F1, and confusion. Otherwise show checked-in representative evidence without remote inference. Inspect class-level regressions: the recorded fine-tuned `TennisSwing` recall was only 0.40 even though aggregate accuracy was flat.

In [ ]:
def parse_activity(value,labels):
    try: candidate=json.loads(value).get("activity","")
    except json.JSONDecodeError: candidate=value
    return norm_label(candidate,labels)
def norm_label(value,labels):
    cleaned=value.strip().strip("`\"' .").casefold()
    for label in labels:
        if cleaned==label.casefold() or label.casefold() in cleaned:return label
    return value.strip()
def classify_video(client,deployment,path,labels):
    uris=[image_uri(im,None,90) for im in extract_frames(path,3)]; return parse_activity(vision_chat(client,deployment,video_system,"These are ordered frames from one clip.",uris,80),labels)
def evaluate_video(holdout):
    base=os.getenv("FOUNDRY_VIDEO_BASE_DEPLOYMENT",""); ft=os.getenv("FOUNDRY_VIDEO_FINE_TUNED_DEPLOYMENT","")
    if not(PROJECT_ENDPOINT and base and ft): raise RuntimeError("Live evaluation requires the project endpoint and both video deployments.")
    labels=sorted(holdout.label.unique()); rows=[]
    with project_clients(PROJECT_ENDPOINT) as (_,client):
        for sample in holdout.itertuples(index=False):
            for kind,deployment in (("base",base),("fine_tuned",ft)):
                pred=classify_video(client,deployment,ucf_root/sample.clip_path,labels); rows.append({"clip_path":sample.clip_path,"label":sample.label,"model_kind":kind,"prediction":pred,"correct":pred==sample.label})
    out=pd.DataFrame(rows); out.to_csv(OUTPUTS/"video-action-recognition"/"holdout-predictions.csv",index=False)
    for kind,g in out.groupby("model_kind"):
        print(kind,accuracy_score(g.label,g.prediction)); print(classification_report(g.label,g.prediction,labels=labels,zero_division=0)); display(pd.DataFrame(confusion_matrix(g.label,g.prediction,labels=labels),index=labels,columns=labels))
    return out
if FOUNDRY_RUN_LIVE_EVALUATION:
    video_evaluation=evaluate_video(video_data["holdout"])
else:
    print("Live evaluation disabled; showing checked-in representative evidence without remote inference.")
    recorded=json.loads((ROOT/"assets"/"recorded-results.json").read_text(encoding="utf-8"))
    video_recorded=recorded["video_action_recognition"]; display(pd.DataFrame([{k:v for k,v in video_recorded.items() if k!="fine_tuned_per_class_recall"}])); display(pd.Series(video_recorded["fine_tuned_per_class_recall"],name="recorded_fine_tuned_recall")); display(Image.open(ROOT/"assets"/"frame-samples.png"))

## Conclusion and next step

The checked-in representative bounded run remained flat at 73.33% overall, while fine-tuned recall ranged from 1.00 for Basketball to 0.40 for TennisSwing. Aggregate accuracy hid class-level weakness. Live service preprocessing can also skip image rows for safety-policy reasons, so review the emitted preprocessing summaries and resulting class balance before accepting a model. Keep live base-versus-fine-tuned inference opt-in, preserve clip boundaries, inspect confusion and recall, and evaluate more clips before accepting a model. Next, continue to `../../03-core-rft`.

> **Variability:** Video accuracy and safety acceptance vary with clip sampling, frame count, privacy preprocessing, model/deployment version, and service conditions. Actual results may vary by model version, data, configuration, region availability, and service conditions.